In [1]:
# =========================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & ĐƯỜNG DẪN
# =========================================================
!pip install -q "huggingface_hub>=0.26.2" polars albumentations pyarrow
!rm -rf /kaggle/working/layout_data/rukopys

In [2]:
import os
import json
import random
import shutil
import re
from pathlib import Path
from tqdm import tqdm
import yaml
from huggingface_hub import hf_hub_download

# --- CẤU HÌNH ĐƯỜNG DẪN ---
GOLD_DIR = Path("/kaggle/input/datasets/notpitomon/htd-train-new/rukopys_augmented/train")
SILVER_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/silver")

OUT_ROOT = Path("/kaggle/working/layout_data/rukopys")
(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

In [3]:
# =========================================================
# BƯỚC 2: HÀM HỖ TRỢ & LOGIC DỮ LIỆU
# =========================================================
TYPE2ID = {
    "handwritten": 0, "printed": 1, "formula": 2, "table": 3, 
    "annotation": 4, "image": 5, "graph": 6
}

def convert_bbox_coco_to_yolo(x, y, w, h, img_w, img_h):
    cx = (x + w / 2.0) / img_w
    cy = (y + h / 2.0) / img_h
    nw = w / img_w
    nh = h / img_h
    return cx, cy, nw, nh

def safe_link(src, dst):
    if not dst.exists():
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copy2(src, dst)

train_files = []
val_files = []

print("🚀 Bắt đầu xử lý dữ liệu...")

# ---------------------------------------------------------
# XỬ LÝ TẬP GOLD (AUGMENTED TRAIN) - CHỐNG RÒ RỈ DỮ LIỆU
# ---------------------------------------------------------
gold_metadata = []
with open(GOLD_DIR / "metadata.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        gold_metadata.append(json.loads(line))

base_names = set()
for d in gold_metadata:
    fname = Path(d["file_name"]).name
    base = re.sub(r'_aug\d+', '', fname)
    base_names.add(base)

base_names = list(base_names)
random.seed(42)
random.shuffle(base_names)

split_idx = int(len(base_names) * 0.85)
train_bases = set(base_names[:split_idx])
val_bases = set(base_names[split_idx:])

print(f"🥇 Số lượng ảnh gốc tập Gold: {len(base_names)} (Train: {len(train_bases)} | Val: {len(val_bases)})")

for d in tqdm(gold_metadata, desc="Xử lý tập Gold"):
    fname = Path(d["file_name"]).name
    base = re.sub(r'_aug\d+', '', fname)
    is_val = base in val_bases
    
    if is_val and "_aug" in fname:
        continue
        
    img_w, img_h = d["image_width"], d["image_height"]
    regions = d.get("regions", [])
    
    in_path = GOLD_DIR / "images" / fname
    if not in_path.exists() or not regions: continue
    
    out_img = OUT_ROOT / "images" / fname
    safe_link(in_path, out_img)
    
    label_lines = []
    for r in regions:
        t = r["type"]
        if t not in TYPE2ID: continue
        x, y, w, h = r["bbox"]
        
        x = max(0.0, float(x))
        y = max(0.0, float(y))
        w = max(1.0, min(float(w), img_w - x))
        h = max(1.0, min(float(h), img_h - y))
        
        cx, cy, nw, nh = convert_bbox_coco_to_yolo(x, y, w, h, img_w, img_h)
        label_lines.append(f"{TYPE2ID[t]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        
    with open(OUT_ROOT / "labels" / (Path(fname).stem + ".txt"), "w") as f:
        f.write("\n".join(label_lines))
        
    if is_val:
        val_files.append(fname)
    else:
        train_files.append(fname)

# ---------------------------------------------------------
# XỬ LÝ TẬP SILVER BẰNG THUẬT TOÁN THAM LAM (GREEDY BALANCING)
# ---------------------------------------------------------
if SILVER_DIR.exists():
    print("🥈 Đang phân tích các class trong tập Silver...")
    valid_silver = []
    with open(SILVER_DIR / "metadata.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            if d.get("regions"):
                valid_silver.append(d)

    # 1. Phân loại ảnh theo class mà nó chứa
    images_by_class = {k: [] for k in TYPE2ID.keys()}
    for i, d in enumerate(valid_silver):
        classes_in_img = set(r["type"] for r in d["regions"] if r["type"] in TYPE2ID)
        for c in classes_in_img:
            images_by_class[c].append(i)

    # Đảo lộn danh sách để lấy ngẫu nhiên
    for c in images_by_class:
        random.shuffle(images_by_class[c])

    SILVER_SAMPLES = 1200
    selected_indices = set()
    current_box_counts = {k: 0 for k in TYPE2ID.keys()}

    # 2. Bắt đầu thuật toán Tham lam chọn 1200 ảnh
    while len(selected_indices) < SILVER_SAMPLES:
        # Lọc ra các class còn ảnh trong kho
        available_classes = [c for c in TYPE2ID.keys() if images_by_class[c]]
        if not available_classes: break # Hết sạch ảnh
        
        # Tìm class đang có ÍT BOX NHẤT hiện tại
        available_classes.sort(key=lambda c: current_box_counts[c])
        target_class = available_classes[0]
        
        # Bốc 1 ảnh chứa class đó ra
        img_idx = images_by_class[target_class].pop()
        
        # Đảm bảo ảnh chưa bị lấy (vì 1 ảnh có thể nằm trong nhiều list class)
        while img_idx in selected_indices and images_by_class[target_class]:
            img_idx = images_by_class[target_class].pop()
            
        if img_idx in selected_indices: continue
            
        # Đưa ảnh vào danh sách chọn
        selected_indices.add(img_idx)
        
        # Cập nhật số lượng box
        for r in valid_silver[img_idx]["regions"]:
            t = r["type"]
            if t in current_box_counts:
                current_box_counts[t] += 1

    silver_sampled = [valid_silver[i] for i in selected_indices]

    print(f"📊 Phân bổ Box trong {len(silver_sampled)} ảnh Silver vừa chọn:")
    for cls, count in sorted(current_box_counts.items(), key=lambda item: item[1]):
        print(f"   - {cls}: {count} boxes")

    # 3. Ghi dữ liệu Silver ra thư mục Train
    for d in tqdm(silver_sampled, desc="Tiến hành gộp Silver vào Train"):
        fname = Path(d["file_name"]).name
        img_w, img_h = d["image_width"], d["image_height"]
        regions = d.get("regions", [])
        
        in_path = SILVER_DIR / "images" / fname
        if not in_path.exists(): continue
            
        out_img = OUT_ROOT / "images" / fname
        safe_link(in_path, out_img)
        
        label_lines = []
        for r in regions:
            t = r["type"]
            if t not in TYPE2ID: continue
            
            # Đổi bbox [x1, y1, x2, y2] sang [cx, cy, w, h] chuẩn YOLO
            x1, y1, x2, y2 = r["bbox"]
            w = x2 - x1
            h = y2 - y1
            
            x1 = max(0.0, float(x1))
            y1 = max(0.0, float(y1))
            w = max(1.0, min(float(w), img_w - x1))
            h = max(1.0, min(float(h), img_h - y1))
            
            cx, cy, nw, nh = convert_bbox_coco_to_yolo(x1, y1, w, h, img_w, img_h)
            label_lines.append(f"{TYPE2ID[t]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            
        with open(OUT_ROOT / "labels" / (Path(fname).stem + ".txt"), "w") as f:
            f.write("\n".join(label_lines))
            
        train_files.append(fname)

# ---------------------------------------------------------
# GHI FILE TEXT & YAML
# ---------------------------------------------------------
random.shuffle(train_files) 

train_txt = OUT_ROOT / "train.txt"
val_txt   = OUT_ROOT / "val.txt"

with open(train_txt, "w") as f:
    for fname in train_files:
        f.write(str(OUT_ROOT / "images" / fname) + "\n")
        
with open(val_txt, "w") as f:
    for fname in val_files:
        f.write(str(OUT_ROOT / "images" / fname) + "\n")

print(f"\n✅ Thống kê cuối cùng: TRAIN có {len(train_files)} ảnh | VAL có {len(val_files)} ảnh")

🚀 Bắt đầu xử lý dữ liệu...
🥇 Số lượng ảnh gốc tập Gold: 1330 (Train: 1130 | Val: 200)


Xử lý tập Gold: 100%|██████████| 3096/3096 [00:10<00:00, 305.40it/s]


🥈 Đang phân tích các class trong tập Silver...
📊 Phân bổ Box trong 1200 ảnh Silver vừa chọn:
   - graph: 39 boxes
   - image: 437 boxes
   - table: 545 boxes
   - annotation: 1483 boxes
   - formula: 3474 boxes
   - printed: 4515 boxes
   - handwritten: 11351 boxes


Tiến hành gộp Silver vào Train: 100%|██████████| 1200/1200 [00:04<00:00, 278.69it/s]


✅ Thống kê cuối cùng: TRAIN có 3853 ảnh | VAL có 200 ảnh


In [4]:
# =========================================================
# BƯỚC 3: SETUP DOCLAYOUT-YOLO & HUẤN LUYỆN
# =========================================================
import sys
from types import ModuleType

%cd /kaggle/working
if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git
%cd DocLayout-YOLO
!pip install -q -e .

checks_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/checks.py"
if os.path.exists(checks_file):
    with open(checks_file, "r", encoding="utf-8") as f: code = f.read()
    if "def check_amp(model):" in code and "return True" not in code.split("def check_amp(model):")[1][:20]:
        code = code.replace("def check_amp(model):", "def check_amp(model):\n    return True\n")
        with open(checks_file, "w", encoding="utf-8") as f: f.write(code)

!pip uninstall ray -y -q
if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

CKPT_PATH = hf_hub_download(repo_id="juliozhao/DocLayout-YOLO-DocStructBench", filename="doclayout_yolo_docstructbench_imgsz1024.pt")

yaml_path = "/kaggle/working/rukopys_dataset.yaml"
data_config = {
    "path": str(OUT_ROOT),
    "train": "train.txt",   
    "val": "val.txt",       
    "nc": 7,
    "names": ["handwritten", "printed", "formula", "table", "annotation", "image", "graph"]
}
with open(yaml_path, "w") as f: yaml.dump(data_config, f, sort_keys=False)
shutil.copy(yaml_path, yaml_path + ".yaml")

print("\n🔥 KHỞI ĐỘNG CỖ MÁY HUẤN LUYỆN 🔥")
!WANDB_MODE=disabled RAY_DISABLE_TUNE=1 python train.py \
  --data /kaggle/working/rukopys_dataset.yaml \
  --model doclayout_yolo_small \
  --epoch 45 \
  --image-size 1280 \
  --batch-size 8 \
  --project rukopys_ft_yolo_small \
  --optimizer Adam \
  --lr0 0.002 \
  --warmup-epochs 1.0 \
  --patience 10 \
  --pretrain {CKPT_PATH} \
  --device 0,1 \
  --workers 8

/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 237 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 21.28 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for doclayout_yolo (pyproject.toml) ... done


doclayout_yolo_docstructbench_imgsz1024.(…):   0%|          | 0.00/40.7M [00:00<?, ?B/s]


🔥 KHỞI ĐỘNG CỖ MÁY HUẤN LUYỆN 🔥
New https://pypi.org/project/doclayout_yolo/0.0.4 available 😃 Update with 'pip install -U doclayout_yolo'
Ultralytics YOLOv0.0.2 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                            CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/root/.cache/huggingface/hub/models--juliozhao--DocLayout-YOLO-DocStructBench/snapshots/8c3299a30b8ff29a1503c4431b035b93220f7b11/doclayout_yolo_docstructbench_imgsz1024.pt, data=/kaggle/working/rukopys_dataset.yaml.yaml, epochs=45, time=None, patience=10, batch=8, imgsz=1280, save=True, save_period=10, val_period=1, cache=False, device=0,1, workers=8, project=rukopys_ft_yolo_small, name=rukopys_dataset.yaml_epoch45_imgsz1280_bs8_pretrain_unknown, exist_ok=False, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=None, amp=True, fraction=1.0,